In [1]:
# Librerías Básicas
%pip install -q ultralytics torchmetrics pycocotools

import shutil
import joblib
import numpy as np
import pandas as pd
import torch
from pathlib import Path

from ultralytics import YOLO, settings
from ultralytics.data.utils import check_det_dataset

# Métricas
from torchmetrics.detection import MeanAveragePrecision


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\sanie\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [5]:
# 2. Cargar datos
# Para entrenar con datos propios, organizá tu carpeta así (formato YOLO):
#
#   mi_dataset/
#     images/train/   foto1.jpg, foto2.jpg ...
#     images/val/     foto3.jpg ...
#     images/test/    foto4.jpg ...
#     labels/train/   foto1.txt, foto2.txt ...   (mismo nombre que la imagen)
#     labels/val/     foto3.txt ...
#     labels/test/    foto4.txt ...
#
# Cada .txt tiene una línea por objeto: clase x_centro y_centro ancho alto
# (valores entre 0 y 1). Ejemplo: 0 0.45 0.60 0.30 0.25
#
# Requisitos: imágenes etiquetadas con cajas (Roboflow, CVAT o LabelImg exportan
# en formato YOLO), ~100 imágenes por clase como mínimo, ~70% train / 20% val / 10% test.
# En Colab: subí la carpeta a Drive o comprimila en .zip y descomprimila con !unzip.

USAR_PROPIO = False   # cambiar a True para usar tu dataset

if USAR_PROPIO:
    RUTA = '/content/mi_dataset'
    class_names = ['clase_0', 'clase_1']   # en el orden de los números de los .txt

    with open(f'{RUTA}/data.yaml', 'w') as f:
        f.write(f"path: {RUTA}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n")
        for i, nombre in enumerate(class_names):
            f.write(f"  {i}: {nombre}\n")

    DATA = f'{RUTA}/data.yaml'
else:
    DATA = 'african-wildlife.yaml'   # dataset conocido, se descarga solo (~100 MB)
    class_names = ['buffalo', 'elephant', 'rhino', 'zebra']

info = check_det_dataset(DATA)   # descarga (si hace falta) y verifica el dataset
TEST = info['test']              # imágenes que ningún modelo ve al entrenar
print('Clases:', class_names)
print('Imágenes de test:', len(list(Path(TEST).glob('*'))))


WARNING ⚠️ Dataset 'african-wildlife.yaml' images not found, missing path '/content/datasets/african-wildlife/images/val'
Unzipping /content/datasets/african-wildlife.zip to /content/datasets/african-wildlife...: 100% ━━━━━━━━━━━━ 3018/3018 3.0Kfiles/s 1.0s
Dataset download success ✅ (4.2s), saved to /content/datasets

Clases: ['buffalo', 'elephant', 'rhino', 'zebra']
Imágenes de test: 227


In [6]:
# 3. Procesar datos
# YOLO no necesita procesar nada a mano: redimensiona, normaliza y hace
# data augmentation solo durante el entrenamiento.

In [7]:
# 4. Configurar TensorBoard
%load_ext tensorboard

settings.update({'tensorboard': True})   # ultralytics escribe los logs solo

shutil.rmtree('runs', ignore_errors=True)

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [8]:
# 5. Inferencia pura: YOLO preentrenado en COCO, sin entrenar con nuestros datos
modelo_base = YOLO('yolo11n.pt')

print('Clases que conoce el modelo:', len(modelo_base.names))
for nombre in class_names:
    print(f'  {nombre}:', 'sí' if nombre in modelo_base.names.values() else 'NO')

Clases que conoce el modelo: 80
  buffalo: NO
  elephant: sí
  rhino: NO
  zebra: sí


In [ ]:
# 6. Transfer Learning: modelo preentrenado con las primeras 10 capas congeladas
# Solo se entrenan las capas finales, que aprenden nuestras 4 clases
modelo_transfer = YOLO('yolo11n.pt')

modelo_transfer.train(data=DATA, epochs=10, freeze=10, patience=5, name='transfer')

In [ ]:
# 7. Fine-Tuning: partir del modelo de transfer, descongelar todo y reentrenar con LR bajo
modelo_ft = YOLO(modelo_transfer.trainer.best)   # ruta al mejor modelo de transfer

modelo_ft.train(data=DATA, epochs=10, optimizer='AdamW', lr0=1e-4, warmup_epochs=0, patience=5, name='finetune')

In [ ]:
# Visualizar todas las corridas en TensorBoard (transfer vs. fine-tuning)
%tensorboard --logdir runs

In [ ]:
# 8. Evaluar los modelos sobre el set de test
# Una sola función para los tres modelos, así la comparación es justa.
# Las clases del modelo se traducen a las del dataset por nombre
# (por ejemplo 'elephant' de COCO -> 'elephant' de nuestro dataset).

def evaluar(modelo):
    mapa = {i: class_names.index(n) for i, n in modelo.names.items() if n in class_names}
    metrica = MeanAveragePrecision(class_metrics=True)

    for r in modelo.predict(TEST, conf=0.001, max_det=100, stream=True, verbose=False):
        # Predicciones (solo clases que existen en nuestro dataset)
        cls = r.boxes.cls.int().tolist()
        ok = [k for k, c in enumerate(cls) if c in mapa]
        pred = {'boxes': r.boxes.xyxy[ok].cpu(),
                'scores': r.boxes.conf[ok].cpu(),
                'labels': torch.tensor([mapa[cls[k]] for k in ok], dtype=torch.long)}

        # Etiquetas reales (del .txt, pasadas a píxeles)
        h, w = r.orig_shape
        txt = Path(r.path.replace('/images/', '/labels/')).with_suffix('.txt')
        e = np.loadtxt(txt, ndmin=2) if txt.exists() and txt.read_text().strip() else np.zeros((0, 5))
        xyxy = np.c_[e[:, 1] - e[:, 3] / 2, e[:, 2] - e[:, 4] / 2,
                     e[:, 1] + e[:, 3] / 2, e[:, 2] + e[:, 4] / 2] * [w, h, w, h]
        real = {'boxes': torch.tensor(xyxy, dtype=torch.float32),
                'labels': torch.tensor(e[:, 0], dtype=torch.long)}

        metrica.update([pred], [real])

    m = metrica.compute()
    fila = {'mAP50': float(m['map_50']), 'mAP50-95': float(m['map']), 'Recall': float(m['mar_100'])}
    for c, v in zip(m['classes'].tolist(), m['map_per_class'].tolist()):
        fila[f'mAP {class_names[c]}'] = max(v, 0.0)
    return fila

modelo_final = YOLO(modelo_ft.trainer.best)

tabla = pd.DataFrame({
    'Inferencia pura': evaluar(modelo_base),
    'Transfer Learning': evaluar(YOLO(modelo_transfer.trainer.best)),
    'Fine-Tuning': evaluar(modelo_final),
}).T.round(3)

tabla

In [ ]:
# 9. Predecir una imagen nueva (un rinoceronte: clase que COCO no conoce)
imagen_nueva = f'{TEST}/3 (12).jpg'   # o la ruta a tu propia imagen

for nombre, m in [('Inferencia pura', modelo_base), ('Fine-Tuning', modelo_final)]:
    resultado = m.predict(imagen_nueva, verbose=False)[0]
    print(nombre)
    for caja in resultado.boxes:
        print('  Clase predicha:', resultado.names[int(caja.cls)], '| Probabilidad:', round(float(caja.conf), 3))
    resultado.show()

In [ ]:
# El modelo de YOLO se guarda con su propio método .save() (no con joblib)
modelo_final.save('nombre_modelo.pt')

# Las clases sí se guardan con joblib
joblib.dump({'class_names': class_names}, 'nombre_modelo_extras.joblib')

print('Modelo y extras guardados correctamente.')

In [ ]:
# # En otro notebook o aplicación cargar el modelo guardado

# modelo_cargado = YOLO('nombre_modelo.pt')
# extras = joblib.load('nombre_modelo_extras.joblib')
# class_names = extras['class_names']

# print('Modelo cargado correctamente.')
# print('Clases esperadas:', class_names)

# resultado = modelo_cargado.predict('mi_foto.jpg')[0]
# for caja in resultado.boxes:
#     print('Clase predicha:', resultado.names[int(caja.cls)])
#     print('Probabilidad:', float(caja.conf))